# Sun–Earth Lagrange Points

Massless satellite at L2 and L4, integrated with `NBodyGravity`, plotted in the rotating frame.

**Expected:** L2 unstable (departs); L4 stable (stays). A small $v_x$ kick ejects L2 sooner and starts a short tadpole arc at L4.


In [ ]:
from nbodygravity import NBodyGravity
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import newton

M_SUN = 1.0            # Solar mass
M_EARTH = 3.003e-6     # Earth mass (in solar masses)
R_EARTH_SUN = 1.0      # Earth–Sun distance (AU)
G = 4 * np.pi**2       # AU^3 / (Msun year^2)

# Angular frequency for a circular orbit with separation R_EARTH_SUN
OMEGA = np.sqrt(G * (M_SUN + M_EARTH) / R_EARTH_SUN**3)

# Center of mass at the origin
mu = M_EARTH / (M_SUN + M_EARTH)
X1_SUN_COM = -mu * R_EARTH_SUN
X2_EARTH_COM = (1.0 - mu) * R_EARTH_SUN

r0_sun_inertial = np.array([X1_SUN_COM, 0.0, 0.0])
r0_earth_inertial = np.array([X2_EARTH_COM, 0.0, 0.0])

# Circular velocities around the CoM: v = omega × r  →  (0, Ω x, 0)
v0_sun_inertial = np.array([0.0, OMEGA * X1_SUN_COM, 0.0])    # X1 < 0 → vy < 0
v0_earth_inertial = np.array([0.0, OMEGA * X2_EARTH_COM, 0.0])  # X2 > 0 → vy > 0


def solve_l2_x(mu, tol=1e-14, max_iter=100):
    """Solve the exact L2 quintic for the barycentric x-coordinate.

    Root q = r/R of:
        q^5 + (3-mu)q^4 + (3-2mu)q^3 - mu q^2 - 2mu q - mu = 0
    then x = (1 - mu) + q.
    """

    def poly(q):
        return (q**5 + (3.0 - mu) * q**4 + (3.0 - 2.0 * mu) * q**3
                - mu * q**2 - 2.0 * mu * q - mu)

    def dpoly(q):
        return (5.0 * q**4 + 4.0 * (3.0 - mu) * q**3 + 3.0 * (3.0 - 2.0 * mu) * q**2
                - 2.0 * mu * q - 2.0 * mu)

    q0 = (mu / 3.0) ** (1.0 / 3.0)
    q = newton(poly, x0=q0, fprime=dpoly, tol=tol, maxiter=max_iter)
    return (1.0 - mu) + q


# L2: beyond Earth on the Sun–Earth line
r0_L2_inertial = np.array([solve_l2_x(mu), 0.0, 0.0])
v0_L2_inertial = OMEGA * np.array([-r0_L2_inertial[1], r0_L2_inertial[0], 0.0])

# L4: equilateral triangle with Sun and Earth
r0_L4_inertial = np.array([
    (X1_SUN_COM + X2_EARTH_COM) / 2,
    R_EARTH_SUN * np.sqrt(3) / 2,
    0.0,
])
v0_L4_inertial = OMEGA * np.array([-r0_L4_inertial[1], r0_L4_inertial[0], 0.0])

print(f"L2 point: {r0_L2_inertial}")
print(f"L4 point: {r0_L4_inertial}")
print(f"|L4-Sun| = {np.linalg.norm(r0_L4_inertial - r0_sun_inertial):.12f} AU")
print(f"|L4-Earth| = {np.linalg.norm(r0_L4_inertial - r0_earth_inertial):.12f} AU")

M_SATELLITE = 0.0


def simulate_lagrange_point_orbit(
    lagrange_point_pos,
    lagrange_point_vel,
    title_suffix="",
    stability_test=False,
    perturbation_val=0.0,
):
    """Integrate a massless satellite at a Lagrange point; plot in the rotating frame."""
    masses = np.array([M_SUN, M_EARTH, M_SATELLITE])
    r0 = np.array([r0_sun_inertial, r0_earth_inertial, lagrange_point_pos])
    v0 = np.array([v0_sun_inertial, v0_earth_inertial, lagrange_point_vel])

    if stability_test:
        v0[2, 0] += perturbation_val

    sim = NBodyGravity(masses, g_const=G, eps=0.0)

    # 20 years with dt ≈ 0.001 yr
    times = np.linspace(0.0, 20.0, 20001)
    t_inertial, r_hist_inertial, v_hist_inertial = sim.integrate(r0, v0, times)

    r_hist_rotating = np.zeros_like(r_hist_inertial)
    for i, current_time in enumerate(t_inertial):
        theta = OMEGA * current_time
        c, s = np.cos(theta), np.sin(theta)
        rotation_matrix = np.array([
            [c, s, 0.0],
            [-s, c, 0.0],
            [0.0, 0.0, 1.0],
        ])
        for j in range(3):
            r_hist_rotating[i, j] = rotation_matrix @ r_hist_inertial[i, j]

    sat_r = r_hist_rotating[:, 2, :2]
    sun_r = r_hist_rotating[:, 0, :2]
    earth_r = r_hist_rotating[:, 1, :2]

    max_dev = np.max(np.linalg.norm(sat_r - sat_r[0], axis=1))
    print(f"{title_suffix}: max |r - r0| in rotating frame = {max_dev:.6e} AU")

    plt.figure(figsize=(9, 9))
    plt.plot(sat_r[:, 0], sat_r[:, 1], "k-", linewidth=2.0, label="Satellite Orbit")
    plt.plot(sun_r[0, 0], sun_r[0, 1], "o", markersize=10, color="orange", label="Sun")
    plt.plot(earth_r[0, 0], earth_r[0, 1], "o", markersize=5, color="blue", label="Earth")
    plt.plot(sat_r[0, 0], sat_r[0, 1], "rx", markersize=8, label="Initial Satellite Pos")
    plt.xlabel("x (AU)")
    plt.ylabel("y (AU)")
    plt.title(f"Satellite Orbit at {title_suffix} (Rotating Frame)")
    plt.gca().set_aspect("equal", adjustable="box")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()


print("--- Simulating L2 point ---")
simulate_lagrange_point_orbit(r0_L2_inertial, v0_L2_inertial, title_suffix="L2")

print("\n--- Simulating L4 point ---")
simulate_lagrange_point_orbit(r0_L4_inertial, v0_L4_inertial, title_suffix="L4")

print("\n--- Simulating L2 with a small perturbation ---")
perturbation = 5e-4  # AU/yr
simulate_lagrange_point_orbit(
    r0_L2_inertial,
    v0_L2_inertial,
    title_suffix=f"L2 with Perturbation (vx += {perturbation} AU/yr)",
    stability_test=True,
    perturbation_val=perturbation,
)

print("\n--- Simulating L4 with a small perturbation ---")
simulate_lagrange_point_orbit(
    r0_L4_inertial,
    v0_L4_inertial,
    title_suffix=f"L4 with Perturbation (vx += {perturbation} AU/yr)",
    stability_test=True,
    perturbation_val=perturbation,
)


## Physical interpretation

- **L2** is a saddle in the effective potential: an equilibrium exists, but it is unstable, so the probe leaves (JWST-like points need station-keeping).
- **L4** is linearly stable for the Earth–Sun mass ratio, so the probe remains co-orbital (~60° ahead of Earth), analogous to Trojan asteroids.
- This model is the **circular restricted three-body problem**, not the full solar system.
